In [1]:
#!pip install pandas
#!pip install seaborn
#!pip install openpyxl
#!pip install yfinance

In [2]:
import os
import csv
import pandas as pd
import numpy as np
from matplotlib import pyplot
import math

In [3]:

df = pd.DataFrame()
header_first = True
mode='w'

for root, dirs, files in os.walk('results'):
    for f in files: 
        if(f.startswith('Result_')):
          file_name=os.path.join(root, f)
          model = f.split('_')[1]
          nm_dataset = f.split('_')[2].split('.')[0]
          dataset = pd.read_csv(file_name, sep =';', encoding = 'latin1', decimal='.')
          dataset['model']=model
          dataset['dataset']=nm_dataset
          dataset['dataset'].replace(['sunspot','canadian','IBM','TSLA'],['Sunspot Numbers','Canadian Lynx','IBM Stock prices','TSLA Stock prices'],inplace=True)          
          dataset.columns=['Dataset','Best Params','n_time_steps','MSE', 'RMSE', 'MAE','MAPE','sMAPE','Duration','model','dataset']
          dataset.to_csv('resume_results.csv',sep=';',mode=mode,index=False, header=header_first, decimal='.')
          header_first = False
          mode='a'
        


C:\Users\Edmilson\AppData\Local\Temp\ipykernel_23152\2152796041.py:14: ChainedAssignmentError: A value is being set on a copy of a DataFrame or Series through chained assignment using an inplace method.
Such inplace method never works to update the original DataFrame or Series, because the intermediate object on which we are setting values always behaves as a copy (due to Copy-on-Write).

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' instead, to perform the operation inplace on the original object, or try to avoid an inplace operation using 'df[col] = df[col].method(value)'.

See the documentation for a more detailed explanation: https://pandas.pydata.org/pandas-docs/stable/user_guide/copy_on_write.html
  dataset['dataset'].replace(['sunspot','canadian','IBM','TSLA'],['Sunspot Numbers','Canadian Lynx','IBM Stock prices','TSLA Stock prices'],inplace=True)
C:\Users\Edmilson\AppData\Local\Temp\ipykernel_23152\2152796041.py

In [4]:
dataset = pd.read_csv(f'resume_results.csv', sep =';', encoding = 'latin1', decimal='.')
#dataset['MAE']=round(dataset['MAE'].str.replace(',','.').astype(float),2)
#dataset['MSE']=round(dataset['MSE'].str.replace(',','.').astype(float),2)
#dataset['RMSE']=round(dataset['RMSE'].str.replace(',','.').astype(float),2)
#dataset['MAPE']=round(dataset['MAPE'].str.replace(',','.').astype(float),2)
#dataset['sMAPE']=round(dataset['sMAPE'].str.replace(',','.').astype(float),2)
#dataset['Duration']=round(dataset['Duration'].str.replace(',','.').astype(float),1)

fs=dataset.pop('dataset')
dataset.insert(1, 'dataset', fs)

fs=dataset.pop('model')
dataset.insert(1, 'model', fs)

fs=dataset.pop('Dataset')

fs=dataset.pop('Best Params')
dataset.insert(9, 'Best Params', fs)


idx = dataset.groupby(['dataset'])['sMAPE'].transform(min)==dataset['sMAPE']
dataset[idx].sort_values(['dataset']).to_csv('best_sMAPE_model_per_dataset.csv',sep=';',decimal=',', index=False)
ds_sMAPE=dataset[idx].sort_values(['dataset'])

idx = dataset.groupby(['dataset'])['MAPE'].transform(min)==dataset['MAPE']
dataset[idx].sort_values(['dataset']).to_csv('best_MAPE_model_per_dataset.csv',sep=';',decimal=',', index=False)
ds_MAPE=dataset[idx].sort_values(['dataset'])

ds_MAPE
#dataset[idx].sort_values(['dataset'])
#dataset[idx].query('freq=="2-Day"').sort_values(['SK_PONTO','freq']).to_csv('tables/best_model_per_reservior_day.csv',sep=';',decimal=',', index=False)
#dataset[idx].query('freq=="1-Hour"').sort_values(['SK_PONTO','freq']).to_csv('tables/best_model_per_reservior_hour.csv',sep=';',decimal=',', index=False)
#dataset[idx].query('freq=="3-Week"').sort_values(['SK_PONTO','freq']).to_csv('tables/best_model_per_reservior_week.csv',sep=';',decimal=',', index=False)
#dataset[idx].query('freq=="2-Day"').sort_values(['SK_PONTO','freq']).head()

,model,dataset,n_time_steps,MSE,RMSE,MAE,MAPE,sMAPE,Duration,Best Params
131,SVR,IBM,2,2.272397,1.507447,0.925102,0.951485,0.48,1.584178,"{'C': 125550, 'epsilon': 0.001, 'gamma': 1e-06..."
177,SVR,TSLA,14,71.402943,8.450026,6.265478,3.201600,1.60,1.676532,"{'C': 12550, 'epsilon': 0.01, 'gamma': 1e-05, ..."
126,SVR,canadian,22,904374.010337,950.985810,344.825348,95.207324,33.31,0.797593,"{'C': 12550, 'epsilon': 0.1, 'gamma': 1e-07, '..."


In [5]:
#Tabela sMAPE dos melhores resultados por Dataset e Modelo
idx = dataset.groupby(['dataset','model'])['sMAPE'].transform(min)==dataset['sMAPE']
dataset[idx].sort_values(['dataset','model']).to_csv('sMAPE_model_per_dataset.csv',sep=';',decimal=',', index=False)
ds_sMAPE=dataset[idx].sort_values(['dataset','sMAPE'])
ds_sMAPE


,model,dataset,n_time_steps,MSE,RMSE,MAE,MAPE,sMAPE,Duration,Best Params
130,SVR,IBM,1,2.353356e+00,1.534065,0.850310,0.964160,0.48,1.601817,"{'C': 125550, 'epsilon': 0.01, 'gamma': 1e-05,..."
131,SVR,IBM,2,2.272397e+00,1.507447,0.925102,0.951485,0.48,1.584178,"{'C': 125550, 'epsilon': 0.001, 'gamma': 1e-06..."
134,SVR,IBM,5,2.344774e+00,1.531265,0.891191,0.964102,0.48,1.457724,"{'C': 12550, 'epsilon': 0.01, 'gamma': 1e-05, ..."
135,SVR,IBM,6,2.392456e+00,1.546757,0.881884,0.971369,0.48,1.584359,"{'C': 125550, 'epsilon': 0.01, 'gamma': 1e-06,..."
70,MLP,IBM,1,8.650831e+01,9.300984,8.276123,6.984947,3.64,23.286870,"{'activation': 'identity', 'alpha': 0.0001, 'h..."
5,ARIMA,IBM,2,3.255978e+02,18.044327,14.453182,12.654717,6.89,37.827930,"ARIMA(1,0,1)(0,0,0)[12] intercept"
29,ARIMA,IBM,2,3.255979e+02,18.044333,14.453199,12.654724,6.89,44.577834,"ARIMA(1,0,1)(0,0,0)[12] intercept"
46,LSTM,IBM,0,4.516344e+02,21.251691,19.319489,16.216335,8.91,103.240686,"LSTM (8,8,8) n_time_steps=0 epochs=200"
49,LSTM,IBM,0,4.516646e+02,21.252402,19.320251,16.216972,8.91,104.816100,"LSTM (8,8,64) n_time_steps=0 epochs=200"
177,SVR,TSLA,14,7.140294e+01,8.450026,6.265478,3.201600,1.60,1.676532,"{'C': 12550, 'epsilon': 0.01, 'gamma': 1e-05, ..."


In [6]:
#Tabela sMAPE dos melhores resultados por Dataset e Modelo
ds = dataset[dataset['model']=='ARIMA']
#ida = ds['sMAPE'].transform(min)==ds['sMAPE']
ds.sort_values(['sMAPE']).to_csv('best_ARIMA_model_per_dataset.csv',sep=';',decimal=',', index=False)
ds.sort_values(['sMAPE'])


,model,dataset,n_time_steps,MSE,RMSE,MAE,MAPE,sMAPE,Duration,Best Params
5,ARIMA,IBM,2,3.255978e+02,18.044327,14.453182,12.654717,6.89,37.827930,"ARIMA(1,0,1)(0,0,0)[12] intercept"
29,ARIMA,IBM,2,3.255979e+02,18.044333,14.453199,12.654724,6.89,44.577834,"ARIMA(1,0,1)(0,0,0)[12] intercept"
20,ARIMA,IBM,17,4.370103e+02,20.904791,18.853031,15.794734,8.67,96.794529,"ARIMA(0,0,1)(0,0,1)[12] intercept"
21,ARIMA,IBM,18,4.377798e+02,20.923189,18.902681,15.803895,8.68,61.138829,"ARIMA(0,0,1)(0,0,1)[12] intercept"
19,ARIMA,IBM,16,4.372087e+02,20.909536,18.827263,15.807119,8.68,274.084701,"ARIMA(0,0,0)(0,0,1)[12] intercept"
18,ARIMA,IBM,15,4.368781e+02,20.901628,18.838200,15.806500,8.68,176.022501,"ARIMA(0,0,0)(0,0,1)[12] intercept"
25,ARIMA,IBM,22,4.405659e+02,20.989662,18.916411,15.855375,8.71,709.973021,"ARIMA(0,0,1)(0,0,1)[12] intercept"
26,ARIMA,IBM,23,4.411455e+02,21.003463,18.703461,15.866834,8.71,231.150749,"ARIMA(0,0,1)(0,0,1)[12] intercept"
27,ARIMA,IBM,24,4.415677e+02,21.013513,18.603913,15.876519,8.72,115.275959,"ARIMA(0,0,1)(0,0,1)[12] intercept"
4,ARIMA,IBM,1,4.374721e+02,20.915833,18.959192,15.914407,8.74,15.583759,"ARIMA(0,0,0)(0,0,0)[12] intercept"


In [7]:
#Tabela sMAPE dos melhores resultados por Dataset e Modelo
ds = dataset[dataset['model']=='SVR']
#ida = ds['sMAPE'].transform(min)==ds['sMAPE']
ds.sort_values(['sMAPE']).to_csv('best_SVR_model_per_dataset.csv',sep=';',decimal=',', index=False)
ds.sort_values(['sMAPE'])


,model,dataset,n_time_steps,MSE,RMSE,MAE,MAPE,sMAPE,Duration,Best Params
131,SVR,IBM,2,2.272397e+00,1.507447,0.925102,0.951485,0.48,1.584178,"{'C': 125550, 'epsilon': 0.001, 'gamma': 1e-06..."
130,SVR,IBM,1,2.353356e+00,1.534065,0.850310,0.964160,0.48,1.601817,"{'C': 125550, 'epsilon': 0.01, 'gamma': 1e-05,..."
134,SVR,IBM,5,2.344774e+00,1.531265,0.891191,0.964102,0.48,1.457724,"{'C': 12550, 'epsilon': 0.01, 'gamma': 1e-05, ..."
135,SVR,IBM,6,2.392456e+00,1.546757,0.881884,0.971369,0.48,1.584359,"{'C': 125550, 'epsilon': 0.01, 'gamma': 1e-06,..."
136,SVR,IBM,7,2.365162e+00,1.537908,0.973746,0.977232,0.49,1.685002,"{'C': 12550, 'epsilon': 0.001, 'gamma': 1e-05,..."
...,...,...,...,...,...,...,...,...,...,...
110,SVR,canadian,6,2.613572e+06,1616.654651,682.102186,145.543444,45.41,1.060087,"{'C': 12550, 'epsilon': 0.001, 'gamma': 1e-08,..."
109,SVR,canadian,5,1.386511e+06,1177.502141,852.909627,205.963371,45.68,1.589678,"{'C': 1255555, 'epsilon': 0.0001, 'gamma': 1e-..."
104,SVR,canadian,0,2.751586e+06,1658.790626,805.127554,153.809913,47.29,13.963204,"{'C': 12550, 'epsilon': 0.1, 'gamma': 1e-05, '..."
145,SVR,sunspot,0,3.391931e+05,582.402830,488.001428,74.007872,70.01,47.789522,"{'C': 12550, 'epsilon': 0.0001, 'gamma': 1e-05..."


In [8]:
#Tabela sMAPE dos melhores resultados por Dataset e Modelo
ds = dataset[dataset['model']=='LSTM']
#ida = ds['sMAPE'].transform(min)==ds['sMAPE']
ds.sort_values(['sMAPE']).to_csv('best_LSTM_model_per_dataset.csv',sep=';',decimal=',', index=False)
ds.sort_values(['sMAPE'])


,model,dataset,n_time_steps,MSE,RMSE,MAE,MAPE,sMAPE,Duration,Best Params
46,LSTM,IBM,0,4.516344e+02,21.251691,19.319489,16.216335,8.91,103.240686,"LSTM (8,8,8) n_time_steps=0 epochs=200"
49,LSTM,IBM,0,4.516646e+02,21.252402,19.320251,16.216972,8.91,104.816100,"LSTM (8,8,64) n_time_steps=0 epochs=200"
48,LSTM,IBM,0,4.520766e+02,21.262093,19.330635,16.225676,8.92,224.269405,"LSTM (8,8,32) n_time_steps=0 epochs=200"
47,LSTM,IBM,0,4.523904e+02,21.269471,19.338539,16.232296,8.92,94.757203,"LSTM (8,8,16) n_time_steps=0 epochs=200"
50,LSTM,IBM,0,4.523937e+02,21.269548,19.338623,16.232368,8.92,334.965133,"LSTM (8,8,100) n_time_steps=0 epochs=200"
55,LSTM,IBM,0,4.547364e+02,21.324548,19.397537,16.281736,8.95,116.747490,"LSTM (8,16,100) n_time_steps=0 epochs=200"
52,LSTM,IBM,0,4.549263e+02,21.329001,19.402306,16.285732,8.96,397.571934,"LSTM (8,16,16) n_time_steps=0 epochs=200"
51,LSTM,IBM,0,4.550676e+02,21.332313,19.405853,16.288704,8.96,351.953900,"LSTM (8,16,8) n_time_steps=0 epochs=200"
54,LSTM,IBM,0,4.550606e+02,21.332150,19.405678,16.288557,8.96,71.548881,"LSTM (8,16,64) n_time_steps=0 epochs=200"
53,LSTM,IBM,0,4.549160e+02,21.328759,19.402046,16.285515,8.96,115.458181,"LSTM (8,16,32) n_time_steps=0 epochs=200"


In [9]:
#Tabela sMAPE dos melhores resultados por Dataset e Modelo
ds = dataset[dataset['model']=='MLP']
#ida = ds['sMAPE'].transform(min)==ds['sMAPE']
ds.sort_values(['sMAPE']).to_csv('best_MLP_model_per_dataset.csv',sep=';',decimal=',', index=False)
ds.sort_values(['sMAPE'])

,model,dataset,n_time_steps,MSE,RMSE,MAE,MAPE,sMAPE,Duration,Best Params
98,MLP,TSLA,2,9.497687e+01,9.745608,6.895231,3.660243,1.83,17.462634,"{'activation': 'identity', 'alpha': 0.001, 'hi..."
99,MLP,TSLA,3,1.068957e+02,10.339038,7.389691,3.855902,1.95,15.293481,"{'activation': 'identity', 'alpha': 0.0001, 'h..."
100,MLP,TSLA,4,1.361626e+02,11.668874,8.282828,4.453805,2.22,12.431246,"{'activation': 'identity', 'alpha': 0.0001, 'h..."
101,MLP,TSLA,5,1.520114e+02,12.329291,8.911302,4.755342,2.37,15.426031,"{'activation': 'identity', 'alpha': 0.0001, 'h..."
97,MLP,TSLA,1,2.225513e+02,14.918153,10.484835,5.276768,2.73,20.008987,"{'activation': 'identity', 'alpha': 0.0001, 'h..."
102,MLP,TSLA,6,2.033976e+02,14.261754,10.311635,5.491994,2.73,12.389855,"{'activation': 'identity', 'alpha': 0.0001, 'h..."
103,MLP,TSLA,7,2.175008e+02,14.747908,10.861445,5.774286,2.87,14.998497,"{'activation': 'identity', 'alpha': 0.001, 'hi..."
70,MLP,IBM,1,8.650831e+01,9.300984,8.276123,6.984947,3.64,23.286870,"{'activation': 'identity', 'alpha': 0.0001, 'h..."
72,MLP,sunspot,1,6.241150e+03,79.000947,29.390993,8.660113,4.43,22.136795,"{'activation': 'identity', 'alpha': 0.0001, 'h..."
86,MLP,sunspot,15,6.533068e+03,80.827394,35.302153,9.071363,4.70,31.776272,"{'activation': 'identity', 'alpha': 0.01, 'hid..."


In [10]:
idx = dataset.groupby(['SK_PONTO','freq','model'])['MAPE'].transform(min)==dataset['MAPE']
dataset[idx].query('model=="SVR-MLP" and freq=="1-Hour"').sort_values(['SK_PONTO','freq']).to_csv('tables/svrmlp_best_model_per_reservior_hour.csv',sep=';',decimal=',', index=False)
dataset[idx].query('model=="SVR-MLP" and freq=="2-Day"').sort_values(['SK_PONTO','freq']).to_csv('tables/svrmlp_best_model_per_reservior_day.csv',sep=';',decimal=',', index=False)
dataset[idx].query('model=="SVR-MLP" and freq=="3-Week"').sort_values(['SK_PONTO','freq']).to_csv('tables/svrmlp_best_model_per_reservior_week.csv',sep=';',decimal=',', index=False)

KeyError: 'SK_PONTO'

In [ ]:
dataset['SVR-LSTM']=np.where(dataset['model']=='SVR-LSTM',dataset['MAPE'],999)
dataset['SVR-MLP']=np.where(dataset['model']=='SVR-MLP',dataset['MAPE'],999)
dataset['SVR']=np.where(dataset['model']=='SVR',dataset['MAPE'],999)
dataset['LSTM']=np.where(dataset['model']=='LSTM',dataset['MAPE'],999)
dataset['ARIMA1']=np.where(dataset['model']=='ARIMA1',dataset['MAPE'],999)
dataset['ARIMA2']=np.where(dataset['model']=='ARIMA2',dataset['MAPE'],999)
dataset['MLP']=np.where(dataset['model']=='MLP',dataset['MAPE'],999)

: 

In [ ]:
idx = dataset.groupby(['SK_PONTO','freq','SVR-LSTM','SVR-MLP','SVR','MLP','LSTM','ARIMA1','ARIMA2'])['MAPE'].transform(min)==dataset['MAPE']
dataset[idx].query('freq=="1-Hour"').sort_values(['SK_PONTO','freq']).to_csv('tables/best_mape_per_reservior_hour.csv',sep=';',decimal=',', index=False)
dataset[idx].query('freq=="2-Day"').sort_values(['SK_PONTO','freq']).to_csv('tables/best_mape_per_reservior_day.csv',sep=';',decimal=',', index=False)
dataset[idx].query('freq=="3-Week"').sort_values(['SK_PONTO','freq']).to_csv('tables/best_mape_per_reservior_week.csv',sep=';',decimal=',', index=False)

: 

In [ ]:
dataset2=dataset.query('freq=="1-Hour"').groupby(['SK_PONTO','freq']).min(['SVR','LSTM','ARIMA1','ARIMA2','MLP','SVR-LSTM','SVR-MLP'])
dataset2

: 

In [ ]:
dataset['SVR-LSTM']=np.where(dataset['model']=='SVR-LSTM',dataset['Duration'],999)
dataset['SVR-MLP']=np.where(dataset['model']=='SVR-MLP',dataset['Duration'],999)
dataset['SVR']=np.where(dataset['model']=='SVR',dataset['Duration'],999)
dataset['LSTM']=np.where(dataset['model']=='LSTM',dataset['Duration'],999)
dataset['ARIMA1']=np.where(dataset['model']=='ARIMA1',dataset['Duration'],999)
dataset['ARIMA2']=np.where(dataset['model']=='ARIMA2',dataset['Duration'],999)
dataset['MLP']=np.where(dataset['model']=='MLP',dataset['Duration'],999)
dataset2=dataset.query('freq=="1-Hour"').groupby(['SK_PONTO','freq']).min(['SVR','LSTM','ARIMA1','ARIMA2','MLP','SVR-LSTM','SVR-MLP'])
dataset2

: 

In [ ]:
idx=dataset.groupby(['SK_PONTO','model','freq','MAPE'])['MAPE'].transform(min)==dataset['MAPE']
dataset[idx].sort_values(['SK_PONTO','freq']).to_csv('results/best_model_per_reservior.csv',sep=';',decimal=',', index=False)

: 